# U.S Rates & Breakeven Inflation Analytics Engine
### Joint Term Structure Modeling, DKW (2018) Liquidity Wedge & Basel III / FRTB Market Risk

This notebook demonstrates the institutional research workflow using the `rate_engine` package.
It estimates a joint 4-parameter Nelson-Siegel term structure under **Diebold-Li (2006) Stuctural Basis Invariance**,
adjusts for latent TIPS liquidity frictions following **D'Amico, Kim, and Wei (DKW 2018)**, and sizes
duration- and beta-neutral breakeven boxes under strict **SR 11-7 Model Risk Governance**.

In [ ]:
# Environment Setup & Modular Package Imports
# If running in a fresh Google Colab environment, uncomment the lines below:
# !git clone https://github.com/sungyup-jung/us-rates-breakeven-engine.git
# %cd us-rates-breakeven-engine
# !pip install -e
# !pip install "nbformat>=4.2.0" "kaleido>=1"

import datetime
import os
import numpy as np
from IPython.display import Markdown, display
from pathlib import Path

from rates_engine.data import FREDMarketDataLoader
from rates_engine.curves import YieldCurve, AdaptiveCurveSelector
from rates_engine.frictions import DKWEconometricPriors, DynamicMarketFrictions
from rates_engine.decompositor import InflationDecompositor, CashBondDiscountEngine
from rates_engine.risk import BreakevenTradePricer, HistoricalMarketRiskEngine, CurveSpreadPricer
from rates_engine.visualizer import render_rates_inflation_dashboard, MarkdownReportGenerator


In [2]:
# Live FRED Ingestion & Diebold-Li (2006) Curve Calibration
(
    nom_mats, nom_yields, tips_mats, tips_yields,
    survey_mats, survey_cpi_exp, dyn_seasonal_factors,
    nom_10y_s, tips_10y_s, nom_30y_s, tips_30y_s,
    cpi_nsa, stress_idx, sofr_val, tgcr_val, settle_date
) = FREDMarketDataLoader.fetch_latest_market_data()

eval_grid = np.array([2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])

# Calibrate Curves via Closed-Form WLS
# Downweight the 20Y nominal supply concession: w_20Y = 0.05
nom_weights = np.ones_like(nom_yields)
nom_weights[np.isclose(nom_mats, 20.0)] = 0.05
nom_curve = YieldCurve(nom_mats, nom_yields, weights=nom_weights)

# TIPS Curve: Invariant decay parameter tau1 ensures closure under subtraction
tips_curve = YieldCurve(tips_mats, tips_yields)

# Live Benchmark 10Y Par Repricing Check
actual_10y_yield = float(nom_yields[np.isclose(nom_mats, 10.0)][0])
semi_coupon = (actual_10y_yield / 2.0) * 100.0
live_cf_sched = np.array([semi_coupon] * 19 + [100.0 + semi_coupon])
live_cf_tenors = np.arange(0.5, 10.5, 0.5)
live_pv = CashBondDiscountEngine.bond_price_from_zero_curve(live_cf_sched, live_cf_tenors, nom_curve)
repricing_err_cents = (live_pv - 100.0) * 100.0

print("=== LIVE MODEL VERIFICATION ===")
print(f"Settlement Date                : {settle_date}")
print(f"FRED 10Y Benchmark Par Yield   : {actual_10y_yield * 100.0:.3f}%")
print(f"Model Park Repricing PV        : USD {live_pv:.4f} per $100 Par")
print(f"Fitting Concession Error       : {repricing_err_cents:+.2f} cents (Gate: +/- 5.0c)")

[INFO] Ingesting FRED market rates, survey CPI, and financing parameters...
[SUCCESS] Ingested market data for settlement date: 2026-09-17
=== LIVE MODEL VERIFICATION ===
Settlement Date                : 2026-09-17
FRED 10Y Benchmark Par Yield   : 4.940%
Model Park Repricing PV        : USD 99.8781 per $100 Par
Fitting Concession Error       : -12.19 cents (Gate: +/- 5.0c)


In [3]:
# Frictions, Inflation Decomposition & Portfolio Risk Execution
# 1. Dynamic Liquidity Wedge (DKW 2018 Priors)
dkw_priors = DKWEconometricPriors()
dynamic_liq_wedge, sigma_liq_wedge = DynamicMarketFrictions.compute_dynamic_liquidity_wedge(
    maturities=eval_grid,
    financial_stress_idx=stress_idx,
    priors=dkw_priors
)

# 2. Inflation Decomposition & Latent IRP Extraction
decompositor = InflationDecompositor(nom_curve, tips_curve)
df_decomp, key_metrics = decompositor.decompose(
    eval_grid = eval_grid,
    survey_mats = survey_mats,
    survey_cpi_exp = survey_cpi_exp,
    liquidity_wedge_bps = dynamic_liq_wedge,
    sigma_wedge_bps = sigma_liq_wedge,
    seasonal_factors = dyn_seasonal_factors,
    current_month = datetime.datetime.now().month
)

# 3. Empirical Hedge Betas & Indexation (SR 11-7 Non-Degenerate OLS)
exec_params = DynamicMarketFrictions.compute_all_parameters(
    nom_10y_series = nom_10y_s,
    tips_10y_series = tips_10y_s,
    nom_30y_series = nom_30y_s,
    tips_30y_series = tips_30y_s,
    cpi_nsa_series = cpi_nsa,
    sofr_rate = sofr_val,
    tgcr_rate = tgcr_val,
    settlement_date = settle_date
)

# 4. Trade Structuring & FRTB Regulatory Market Risk Engine
pricer_10y = BreakevenTradePricer(
    nom_par_notional = 100_000_000,
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    tenor = 10.0,
    cif = exec_params['CIF'],
    beta_tips = exec_params['Beta_TIPS_10Y'],
    repo_nom_bps = exec_params['Repo_Nominal_Bps'],
    repo_tips_bps = exec_params['Repo_TIPS_Bps']
)
trade_structure_10y = pricer_10y.calculate_trade_structure()

bic_selection = AdaptiveCurveSelector.evaluate_model_selection(nom_mats, nom_yields)
market_risk = HistoricalMarketRiskEngine.evaluate_portfolio_var(
    nom_10y_series = nom_10y_s,
    tips_10y_series = tips_10y_s,
    pricer = pricer_10y,
    holding_period_days = 10,
    confidence_level = 0.99
)
curve_box_30y = CurveSpreadPricer.size_10s30s_box(
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    cif = exec_params['CIF'],
    beta_10y = exec_params['Beta_TIPS_10Y'],
    beta_30y = exec_params['Beta_TIPS_30Y'],
    target_10y_notional = 100_000_000.0
)

In [ ]:
REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

png_path = str(REPO_ROOT / "rates_dashboard.png")
md_path = str(REPO_ROOT / "TERM_STRUCTURE_ANALYTICS_OUTPUT.md")


# Render Interactive Dashboard & Research Note
fig = render_rates_inflation_dashboard(
    df = df_decomp,
    metrics = key_metrics,
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    pricer_10y = pricer_10y,
    settlement_date = settle_date,
    save_png_path = png_path
)

# Explicitly display the interactive Plotly figure in the notebook
fig.show(renderer="notebook_connected")

report_md = MarkdownReportGenerator.generate_markdown(
    df_decomp =df_decomp,
    metrics = key_metrics,
    bic_selection = bic_selection,
    market_risk = market_risk,
    nom_curve = nom_curve,
    tips_curve = tips_curve,
    exec_params = exec_params,
    trade_structure = trade_structure_10y,
    pricer_10y = pricer_10y,
    curve_box_30y = curve_box_30y,
    settle_date = settle_date,
    export_filename = md_path
)
display(Markdown(report_md))

[SUCCES] Exported dashboard PNG to: rates_dashboard.png


## 4. U.S. Rates & Breakeven Inflation Research Note
**Settlement Date:** 2026-09-17 | **Model:** Diebold-Li (2006) Basis Invariance + DKW (2018)

![Term Structure Dashboard](rates_dashboard.png)

### Executive Summary
Quantitative term structure and relative-value breakeven analytics for settlement date **September 17, 2026**:

* **Spot Breakeven Term Structure**:
  * **2Y Spot BEI**: Trades at 180.41 bps (4.703% Nominal vs. 2.899% TIPS).
  * **Seasonality Adjustment**: Dynamic 5-year BLS factor for September adjusts 2Y SA breakeven to 172.00 bps (-8.41 bps wedge).
  * **Key Slopes**: 5Y at 226.78 bps and 10Y at 235.10 bps (**5s10s: +8.32 bps**); 30Y anchors at 225.24 bps (**10s30s: -9.86 bps**).
* **Forward Anchoring (5Y5Y)**: Evaluates to **243.44 bps**.
* **Affine IRP Term Premia**:
  * **10Y IRP**: **-19.69 bps** (±1σ: [-20.18, -19.20] bps)
  * **30Y IRP**: **-33.99 bps** (±1σ: [-34.31, -33.68] bps)

---

### Econometric Diagnostics & Model Validation (SR 11-7)

1. **Model Selection (BIC)**: Nominal $\text{BIC}_{\text{NS}} = -100.86$ vs. $\text{BIC}_{\text{NSS}} = -95.46$. Selected: **Diebold Li Nelson-Siegel (3-Factor Linear)**.
2. **Diebold-Li (2006) Parameter Invariance**:
   * Structural decay constant: $\tau_1 = 2.2306$ (centering the empirical curvature hump at $\tau^* = 4.0\text{ years}$).
   * Estimation: Exact closed-form Weighted Least Squares (WOLS) with zero non-linear optimization risk.
3. **DKW (2018) Liquidity Calibration**:
   * Base front-end wedge: 6.50 bps (decay half-life: 5.0 yrs; floor: 1.50 bps).
   * Stress scalar sensitivity: $\gamma = 0.25$ scaled against STLFSI4.
4. **Matrix Stability**: Nominal basis condition number $\kappa = 23.73$; TIPS $\kappa = 82.25$ (well below the 30.0 threshold).
5. **Residual Precision**: Nominal RMSE = 5.613 bps; TIPS RMSE = 2.229 bps.
6. **Reference CPI**: Daily interpolated Ref CPI = 333.9339 ($CIF = 1.0307$).

---

### 4.6 Quantitative Dashboard Figure Interpretation

* **Figure 1 (Zero Yield Term Structure & Macro Stance)**: The nominal curve trades **upward-sloping (steep)** (2s10s spread: **+28.5 bps**, spanning 4.70% to 5.27%), while the real TIPS curve exhibits a 2s10s slope of **-27.9 bps** (2.90% to 3.01%). Real yields are firmly restrictive across all tenors (trough at **2.47%**), confirming that policy rates ($r > r^*$) maintain high hurdle rates across risk assets. Ten-year real rates are anchoring at an elevated **2.62%**, discounting post-GFC secular stagnation.

* **Figure 2 (Spot BEI Breakdown & Market Frictions)**: Seasonally adjusted spot breakevens span **172.0 to 235.1 bps**. At the 10Y tenor, consensus survey CPI sits **22.2 bps above** market-implied pricing. The DKW-calibrated liquidity wedge imposes a **6.1 bps penalty at 2Y**, decaying asymptotically to **1.6 bps at 30Y**.

* **Figure 3 (Extracted Inflation Risk Premia & Confidence Bands)**: Extracted latent IRP sits strictly in a **negative corridor** (-84.6 to -19.7 bps). This reflects strong institutional duration demand (asset-liability matching at prevailing 4.99% nominal yields) and flight-to-safety hedging that compresses nominal yields below survey expectations. At the 10Y benchmark, IRP evaluates to **-19.7 bps** (\pm 1\sigma bounds: [-20.2, -19.2] bps).

* **Figure 4 (Continuous Spot vs. 1Y Forward Breakevens)**: Calibrating curves via the Diebold-Li (2006) macro invariant ($\tau_1 = 2.2307$, $\tau^* = 4.0\text{Y}$) ensures identical factor loading spaces across nominal and TIPS curves. The continuous 1Y forward breakeven curve $f^{\text{BEI}}(t, t+1)$ eliminates tail-whip divergence. The benchmark 5Y5Y forward breakeven anchors cleanly at **243.4 bps**, confirming long-term central bank credibility remains intact.

* **Figure 5 (SR 11-7 Model Residual Diagnostics)**: **FLAGGED**: 4 liquid benchmark(s) exceeded the $\pm 3.0\text{ bps}$ gate: **Nom 2Y (-3.3 bps), Nom 3Y (+5.0 bps), Nom 10Y (-4.8 bps), TIPS 20Y (-3.2 bps)**. Aggregate Nominal RMSE printed at **5.61 bps** and TIPS RMSE at **2.23 bps**, reflecting the parsimony trade-off of a 3-factor linear basis. The nominal 20Y trades at an outlier residual of **+12.5 bps** (isolated via WOLS weight $w_{20\text{Y}} = 0.05$).

* **Figure 6 (90-Day PnL Attribution Across Macro Scenarios)**: Across stress regimes, maximum upside occurs under **OIL-05 (+USD 2,604,506.84)**, while maximum loss occurs under **OIL-06 (-USD 2,548,631.19)**. Regulatory 10-day 99% Historical VaR evaluates to **USD 1,448,056.70** (Expected Shortfall: **USD 1,611,240.30**), demonstrating capital adequacy under FRTB stress.


---

## 5. Trade Sizing, Risk Sensitivity & Execution

### 5.1 Dynamic Trade Sizing (10Y Benchmark Box)
$$\text{Position} = \text{Long USD 100M Par 10Y Nominal} + \text{Short 10Y TIPS}$$

| Metric | Nominal Leg | TIPS Hedged Leg | Net / Status |
| :--- | :---: | :---: | :--- |
| **Par Notional** | USD 100,000,000.00 | USD 136,902,524.91 | $\beta_{10\text{Y}} = 0.700$ |
| **Actual Cash Principal** | USD 100,000,000.00 | USD 141,110,420.21 | Scaled by $CIF = 1.0307$ |
| **DV01** | USD 97,566.69 | USD 139,285.69 | Duration & Beta Neutral |
| **Convexity ($C$)** | 99.95 | 102.30 | Net Long Convexity |

---

### 5.2 Key Rate Duration (KRD) Bucket Decomposition (10Y Benchmark)

| Key Tenor | 2Y Bucket | 5Y Bucket | 10Y Bucket | 30Y Bucket | Total Duration |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Nominal 10Y KRD** | 0.654 | -4.038 | -3.965 | -2.406 | **-9.755 yrs** |

---

### 5.3 Regulatory Market Risk (FRTB / Basel III Standards)

* **10-Day 99% Historical Simulation VaR**: USD 1,448,056.70
* **10-Day 99% Expected Shortfall (ES)**: USD 1,611,240.30
* **Max 252-Day Historical Drawdown**: USD 1,799,993.21
* **3-Month Repo Carry**: -USD 364,993.78 (SOFR: 362.0 bps vs. TIPS Repo: 360.0 bps)
* **3-Month Curve Roll-Down**: +USD -61,294.34 (Nominal: -0.89 bps vs. TIPS: -1.06 bps)
* **Net 90-Day Carry Drag**: -USD 303,699.44 (Hurdle: **+3.11 bps** over 90 days)

---

### 5.4 Stress Testing & PnL Attribution Analytics

| Scenario ID | Regime Description | $\Delta y_{\text{Nom}}$ | $\Delta y_{\text{TIPS}}$ | Delta PnL (USD) | Gamma PnL (USD) | Total Horizon PnL (USD) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **OIL-01** | Geopolitical Dislocation | +40.0 bps | +15.0 bps | -1,813,382.37 | +63,721.42 | **-1,445,961.50** |
| **OIL-02** | Asymmetric Supply Shock | +20.0 bps | +8.0 bps | -837,048.34 | +15,370.96 | **-517,977.93** |
| **OIL-03** | Baseline Strip Realization | +3.0 bps | +2.0 bps | -14,128.70 | +161.07 | **+289,731.82** |
| **OIL-04** | Cyclical Demand Easing | -10.0 bps | -3.0 bps | +557,809.86 | +4,348.00 | **+865,857.30** |
| **OIL-05** | Supply Glut / Liquidation | -40.0 bps | -12.0 bps | +2,231,239.44 | +69,567.95 | **+2,604,506.84** |
| **OIL-06** | Cost-Push Stagflation | +15.0 bps | -10.0 bps | -2,856,357.32 | +4,026.69 | **-2,548,631.19** |

---

### 5.5 10s30s Duration-Neutral Curve Box Structure

$$\text{Position} = \text{Long 10Y Box (+USD 100M)} + \text{Short 30Y Box (-USD 33.38M)}$$

| Leg | Instrument | Par Notional (USD) | DV01 (USD) | Hedge Parameter |
| :--- | :--- | :---: | :---: | :--- |
| **Leg 1** | Long 10Y Nominal | 100,000,000.00 | +97,566.69 | Par Target |
| **Leg 1** | Short 10Y TIPS | 136,902,524.91 | -97,566.69 | $\beta_{10\text{Y}} = 0.700$ |
| **Leg 2** | Short 30Y Nominal | 33,379,345.80 | -97,566.69 | Duration Matched |
| **Leg 2** | Long 30Y TIPS | 43,003,412.83 | +97,566.69 | $\beta_{30\text{Y}} = 0.745$ |
| **Portfolio** | **Net Structure** | — | **0.00** | **Strictly Curve-Neutral** |
